# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**Lane:** Content Decline / Refresh Prioritization. Skills loaded per `skills/README.md`: `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

Paper audited: `docs/flyrank-seo-research-march-2026.pdf` — *The State of AI-Driven SEO, March 2026*.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))


30000 pages |  declining rate: 0.542


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---

### Finding #1 — "The Anatomy of Growing Content" (paper p.6, direct aggregate comparison)

**The claim:** growing pages average 3,180 words vs. 2,311 for declining pages (37.6% longer), and are 20% younger (184d vs. 230d). Framed as CONFIRMED, backed by large samples (74.8K up vs. 45.6K down).

**Where the label comes from:** `trend_direction`, computed from 30-day-vs-previous-30-day impression change — the same construction logic my own `is_declining_label` uses (though the paper's threshold is >10%/-10% for up/down, while my starter CSV uses ±20% — a real difference between the two exports worth noting, not just a rounding choice).

**My methodology question:** the paper's own Methodology page discloses, *word for word*, that "content age confounds model-performance comparisons." Finding #1 reports a word-count gap AND an age gap between the same two groups in the same breath — so how much of the 869-word gap survives once you control for age? If growing pages are systematically younger, and older pages in this portfolio were written under an older, shorter editorial standard (a publish-date artifact rather than a decline cause), the word-count comparison could be partly re-measuring the age gap the paper already flags as a confound elsewhere. A same-age-band comparison (e.g. word count for up vs. down pages *within* the 181-270 day age tier only) would isolate the two variables the paper currently reports together.

---

### Finding #2 — "What Predicts Growth?" (paper p.29, ML appendix)

**The claim:** a logistic regression reaches 71% holdout accuracy separating growing from declining pages, with content age as the strongest negative coefficient and days-visible / recent impressions as the strongest positive ones.

**Where the label comes from:** the same `trend_direction`-derived growth/decline split as Finding #1 — an observed outcome, not a proxy, which is good practice.

**My methodology question:** the Methodology page states the ML pipeline uses an "80/20 split" for Random Forest and Logistic Regression, computed across 61.8K content pieces spanning 57 brands — but doesn't say whether that split is grouped by brand. This is exactly the question my own Week-5/6 work had to answer: pages from the same brand share hidden structure (CMS templates, editorial voice, industry vertical), so if a brand's pages can land in both the 80% training slice and the 20% holdout, part of that 71% accuracy could be the model recognizing a brand it has already partly seen, not a general growth/decline signal that would transfer to a 58th brand. Section 2 below shows exactly how large that gap can be on my own dataset — worth the paper disclosing whether its split was random or brand-grouped, the same way it already discloses the age confound.

In [2]:
# Grounding check: my starter export uses a stricter threshold than the paper states for its
# own trend_direction (±20% here vs. the paper's >10%/-10%) -- a real difference between the two
# FlyRank exports, not something I'm assuming.
print(df["trend_direction"].value_counts())
print("\nDeclining rate under this export's definition:", round(df['is_declining_label'].mean(), 3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate under this export's definition: 0.542


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Turning the same question from Finding #2 on myself: my Week-5 notebook already used a client-grouped holdout. Here I build the **naive random row-split version I did NOT publish** as the "before," to show exactly the size of the gap a grouped split closes — the same gap I'm asking FlyRank's paper about above.

In [3]:
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
missing_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                     "engagement_rate", "scroll_rate", "ai_traffic_pct"]
for c in missing_flag_cols:
    df[f"{c}_missing"] = df[c].isna().astype(int)

numeric_frame = (df[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
                 .replace([np.inf, -np.inf], np.nan).fillna(0))
missing_frame = df[[f"{c}_missing" for c in missing_flag_cols]]
cat_frame = df[MODEL_CATEGORICAL_FEATURES].fillna("unknown").astype(str)
cat_dummies = pd.get_dummies(cat_frame, prefix=MODEL_CATEGORICAL_FEATURES, dtype=float)
feature_frame = pd.concat(
    [numeric_frame.reset_index(drop=True), missing_frame.reset_index(drop=True), cat_dummies.reset_index(drop=True)],
    axis=1,
)
y = df["is_declining_label"]
print("Feature matrix:", feature_frame.shape)


Feature matrix: (30000, 60)


In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def fit_and_score(X_train, X_test, y_train, y_test):
    lr = Pipeline([("scaler", StandardScaler()),
                   ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))])
    lr.fit(X_train, y_train)
    lr_scores = lr.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    out = []
    for name, scores in [("logistic_regression", lr_scores), ("random_forest", rf_scores)]:
        out.append({
            "model": name,
            "test_base_rate": y_test.mean(),
            "roc_auc": roc_auc_score(y_test, scores),
            "avg_precision": average_precision_score(y_test, scores),
            "precision@20": precision_at_k(y_test, scores, 20),
            "precision@50": precision_at_k(y_test, scores, 50),
            "precision@100": precision_at_k(y_test, scores, 100),
        })
    return pd.DataFrame(out).set_index("model")

# --- BEFORE: naive random row split (what the paper's methodology page reads like: "80/20 split",
# no grouping mentioned) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    feature_frame, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
before = fit_and_score(X_train_r, X_test_r, y_train_r, y_test_r)
print("BEFORE -- naive random row split:")
before.round(3)


BEFORE -- naive random row split:


,test_base_rate,roc_auc,avg_precision,precision@20,precision@50,precision@100
model,,,,,,
logistic_regression,0.542,0.711,0.727,0.95,0.92,0.9
random_forest,0.542,0.755,0.765,0.85,0.90,0.9


In [5]:
shared_clients = set(df.loc[X_train_r.index, "client_id"]) & set(df.loc[X_test_r.index, "client_id"])
print(f"Clients appearing in BOTH the random train and random test split: "
      f"{len(shared_clients)} of {df['client_id'].nunique()} total clients.")


Clients appearing in BOTH the random train and random test split: 31 of 32 total clients.


In [6]:
# --- AFTER: client-grouped holdout, same as Week 5 ---
client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
after = fit_and_score(feature_frame[~test_mask], feature_frame[test_mask], y[~test_mask], y[test_mask])
print("AFTER -- client-grouped holdout (0 shared clients between train and test, by construction):")
after.round(3)


AFTER -- client-grouped holdout (0 shared clients between train and test, by construction):


,test_base_rate,roc_auc,avg_precision,precision@20,precision@50,precision@100
model,,,,,,
logistic_regression,0.391,0.702,0.524,0.35,0.40,0.43
random_forest,0.391,0.749,0.618,0.80,0.76,0.73


In [7]:
gap = (before[["roc_auc", "avg_precision", "precision@20", "precision@50", "precision@100"]]
       - after[["roc_auc", "avg_precision", "precision@20", "precision@50", "precision@100"]])
print("Gap (naive random MINUS honest client-grouped) -- positive numbers are optimism the naive split added:")
gap.round(3)


Gap (naive random MINUS honest client-grouped) -- positive numbers are optimism the naive split added:


,roc_auc,avg_precision,precision@20,precision@50,precision@100
model,,,,,
logistic_regression,0.009,0.203,0.60,0.52,0.47
random_forest,0.006,0.147,0.05,0.14,0.17


**Reading the gap:** under the naive random split, 31 of 32 clients show up in *both* train and test — almost total group overlap. Logistic Regression's precision@20 jumps from 0.35 (honest) to 0.95 (naive) and Random Forest's from 0.80 to 0.85; ROC-AUC and average precision move the same direction for both models. I'm not calling all of that pure memorization, though — the two test sets also have different base rates (0.542 naive vs. 0.391 grouped) because the six held-out clients in the grouped split genuinely decline less often than the portfolio average, and that's a real fact about deploying to an unseen client, not an artifact. What I can say cleanly: the honest, client-grouped number is the one that answers "will this work on a brand-new client," and it is measurably, sometimes dramatically, lower than the naive number for both models — exactly the question I'm asking FlyRank's own 71%-accuracy claim to answer about its split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
# --- Confession test: inject the literal label-source column and watch the score break toward 1.0.
# If this does NOT jump, the test harness itself is broken (per the skill's "how to verify" step).
X_leak = feature_frame.copy()
X_leak["trend_pct_LEAK"] = df["trend_pct"].fillna(0)

def eval_on_client_split(X):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X[~test_mask], y[~test_mask])
    scores = rf.predict_proba(X[test_mask])[:, 1]
    return roc_auc_score(y[test_mask], scores), average_precision_score(y[test_mask], scores)

honest_auc, honest_ap = eval_on_client_split(feature_frame)
leak_auc, leak_ap = eval_on_client_split(X_leak)
print(f"Honest feature set:        roc_auc={honest_auc:.3f}  avg_precision={honest_ap:.3f}")
print(f"WITH trend_pct injected:   roc_auc={leak_auc:.3f}  avg_precision={leak_ap:.3f}   <- confession")
assert leak_auc > 0.99, "Harness did not catch an obvious label-derived leak -- fix the test, not the model."
print("\nHarness confirmed working: injecting the label-source column collapses the problem to trivial.")


Honest feature set:        roc_auc=0.749  avg_precision=0.618
WITH trend_pct injected:   roc_auc=1.000  avg_precision=1.000   <- confession

Harness confirmed working: injecting the label-source column collapses the problem to trivial.


In [9]:
# --- Overlap-window check: impressions_90d structurally CONTAINS impressions_last_30d and
# impressions_prev_30d, and those two numbers are exactly what trend_pct (the label's source) is
# built from. That's a real, disclosable structural overlap -- not a coding bug like Week 3's
# off-by-one, but the same category of risk: a "feature" that is not fully separable from the
# outcome window. I did not use impressions_last_30d / impressions_prev_30d directly as features,
# but log_impressions_90d (which I DID use) is highly correlated with them.
corr_90d_last30 = df["impressions_90d"].corr(df["impressions_last_30d"])
print(f"corr(impressions_90d, impressions_last_30d) = {corr_90d_last30:.3f}  <- structural overlap, disclosed")

overlap_cols = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]
X_no_overlap = feature_frame.drop(columns=overlap_cols)
with_auc, with_ap = eval_on_client_split(feature_frame)
without_auc, without_ap = eval_on_client_split(X_no_overlap)
print(f"\nWith 90d totals:    roc_auc={with_auc:.3f}  avg_precision={with_ap:.3f}")
print(f"Without 90d totals: roc_auc={without_auc:.3f}  avg_precision={without_ap:.3f}")


corr(impressions_90d, impressions_last_30d) = 0.918  <- structural overlap, disclosed

With 90d totals:    roc_auc=0.749  avg_precision=0.618
Without 90d totals: roc_auc=0.743  avg_precision=0.602


**Verdict:** the confession test works as designed — a known label-derived column collapses the problem instantly (AUC to 1.000), confirming the test harness itself can catch a real leak. The overlap-window check is more nuanced: `impressions_90d` and `impressions_last_30d` correlate at 0.918 — a real, disclosable structural overlap, since the 90-day total literally contains the 30-day window the label is built from. But empirically dropping the four 90-day-total columns only costs about 0.006 AUC and 0.016 average precision — the overlap isn't doing meaningful work for the model. **I'm adopting the more conservative feature set (without the 90-day totals) as my final honest model going forward**, since the small performance cost is worth removing a disclosed-but-avoidable overlap entirely, rather than relying on "it probably isn't leaking much."

**Attack checklist, applied to this notebook:**
- [x] Timeline drawn: every retained feature is a trailing, pre-decision fact about the same content item (age, staleness, position, CTR, word count); `trend_direction` / `trend_pct` never enter the matrix.
- [x] No label-derived or sibling columns in the final features (confirmed by the confession test above, and by the assert that catches it if it weren't).
- [x] No product flags — this starter export doesn't ship FlyRank's own `health_score` / `recommended_action` / optimization-flag columns at all.
- [x] Population selection checked: every row in `content_refresh_anonymized.csv` is included, no filter tied to the outcome window.
- [x] Split grouped by client (Section 2).
- [x] Base rate printed next to every metric (Section 2 tables).
- [x] Top feature importance sanity-checked in Week 5 — no single feature towered over the rest.
- [x] Metrics computed out-of-fold on the held-out client slice, never in-sample.
- [ ] Sealed/holdout receipts — not yet applicable; nothing in this project is claimed as a sealed evaluation.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from `w05_model.ipynb`, Section 3):**
> "Random Forest is the clear winner over Logistic Regression too: roughly double the precision@20/50/100, a meaningfully higher ROC-AUC (0.749 vs 0.702) and average precision (0.618 vs 0.524), and a much better recall/F1 trade-off. I did not need Gradient Boosting to beat the baseline convincingly."

This reads as a settled, general verdict ("clear winner," "did not need"). What I can actually defend, after this week's audit, is narrower: one specific model, on one specific 6-client holdout, using one specific feature set that has since changed slightly (Section 3 dropped the 90-day totals).

**Rewritten:**
> On this client-grouped holdout split, Random Forest **measured** higher precision@20/50/100, ROC-AUC, and average precision than Logistic Regression — a **directional** result, not a guarantee it wins on a different client split or a larger holdout. The gap is large enough to be **decision-support** for preferring Random Forest as the next model to try, but I have not tested Gradient Boosting, and I would not claim Random Forest is the ceiling for this lane.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.